# Plotting training results

This notebook serves as a visualization tool for the training results of the **FlowerLLM** training results.
It reads the metric from the training curves directly from Wandb and collect them in `pandas` data frames ready to be analyzed and plotted.

In [3]:
# Imports
from logging import INFO, ERROR
from typing import Any
import time
import numpy as np
from matplotlib import pyplot as plt
import matplotlib.style as mplstyle
import matplotlib.patches as mpatches
import wandb
import pandas as pd
import seaborn as sns
from matplotlib.ticker import MultipleLocator
import enum
from flwr.common import log

# Change font to respect submission requirements
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = "Times New Roman"
plt.rcParams["font.size"] = 14
plt.rcParams["font.weight"] = "bold"
# Colors and line styles
color_palette = sns.color_palette("colorblind")
line_styles = ["-", "--", ":", "-."]
# Bolded axis labels
plt.rcParams["axes.labelweight"] = "bold"
mplstyle.use("fast")

In [4]:
def add_run_uuid(run_uuid: str) -> None:
    api = wandb.Api(timeout=10000)
    for i in range(64):
        try:
            actual_run_uuid = f"{run_uuid}_client_{i}"
            run = api.run(f"camlsys/pollen-llm/{actual_run_uuid}")
            run.config["run_uuid"] = run_uuid
            run.update()
        except Exception as e:
            log(INFO, e)
    try:
        actual_run_uuid = f"{run_uuid}_server"
        run = api.run(f"camlsys/pollen-llm/{actual_run_uuid}")
        run.config["run_uuid"] = run_uuid
        run.update()
    except Exception as e:
        log(INFO, e)
    try:
        actual_run_uuid = f"{run_uuid}"
        run = api.run(f"camlsys/pollen-llm/{actual_run_uuid}")
        run.config["run_uuid"] = run_uuid
        run.update()
    except Exception as e:
        log(INFO, e)
    try:
        actual_run_uuid = f"{run_uuid}_centralised"
        run = api.run(f"camlsys/pollen-llm/{actual_run_uuid}")
        run.config["run_uuid"] = run_uuid
        run.update()
    except Exception as e:
        log(INFO, e)


def change_run_uuid(old_run_uuid: str, new_run_uuid: str) -> None:
    api = wandb.Api(timeout=10000)
    for i in range(64):
        try:
            actual_run_uuid = f"{old_run_uuid}_client_{i}"
            run = api.run(f"camlsys/pollen-llm/{actual_run_uuid}")
            run.config["run_uuid"] = new_run_uuid
            run.update()
        except Exception as e:
            log(INFO, e)
    try:
        actual_run_uuid = f"{old_run_uuid}_server"
        run = api.run(f"camlsys/pollen-llm/{actual_run_uuid}")
        run.config["run_uuid"] = new_run_uuid
        run.update()
    except Exception as e:
        log(INFO, e)
    try:
        actual_run_uuid = f"{old_run_uuid}"
        run = api.run(f"camlsys/pollen-llm/{actual_run_uuid}")
        run.config["run_uuid"] = new_run_uuid
        run.update()
    except Exception as e:
        log(INFO, e)
    try:
        actual_run_uuid = f"{old_run_uuid}_centralised"
        run = api.run(f"camlsys/pollen-llm/{actual_run_uuid}")
        run.config["run_uuid"] = new_run_uuid
        run.update()
    except Exception as e:
        log(INFO, e)

In [5]:
class CentSuffix(str, enum.Enum):
    """Enum class for suffixes of centralized runs."""

    CENTRALIZED = "_centralised"
    CLIENT_0 = "_client_0"
    NONE = ""

In [6]:
class AxisLabels(str, enum.Enum):
    """Enum class for x- or y-axis labels."""

    FED_ROUND = "Federated Round"
    PPL = "Perplexity"

In [7]:
class PlottingCosmetics(str, enum.Enum):
    """Enum class for plotting cosmetics constants."""

    UPPER_RIGHT_POS = "upper right"
    UPPER_CENTER_POS = "upper center"

In [8]:
# Set the run_id to be retrieved
run_id_dict = {
    "full_participation": {
        # Number of local steps per round
        "64_steps": {
            "8": ("fed-lr-sched1-8cpr64-bs32-20241023_232107", 8, True, False),
            "4": (
                "fed-lr-sched0-4cpr64-bs32-20241022_145655",
                4,
                True,
                False,
            ),  # This has sched0 but it's actually sched1
            "2": ("fed-lr-sched1-2cpr64-bs32-20241023_144602", 2, True, False),
            "1": (
                "fed-lr-sched0-1cpr64-bs32-20241022_135427",
                1,
                True,
                False,
            ),  # This is of course repeated for several experiments as it is always the same
        },
        "128_steps": {
            "8": ("fed-steps-8cpr128-bs32-20241026_004019", 8, True, False),
            "4": ("fed-steps-4cpr128-bs32-20241025_185132", 4, True, False),
            "2": (
                "fed-steps-2cpr128-bs32-20241027_115403",
                2,
                True,
                False,
            ),  # Running on CaMLSys cluster
            "1": ("fed-lr-sched0-1cpr64-bs32-20241022_135427", 1, True, False),
        },
        "512_steps": {
            "16": (
                "fed-steps-16cpr512-bs32-20241027_083734",
                16,
                True,
                False,
            ),
            "8": ("fed-steps-8cpr512-bs32-20241025_151104", 8, True, False),
            "4": ("fed-steps-4cpr512-bs32-20241025_110058", 4, True, False),
            "2": ("fed-steps-2cpr512-bs32-20241026_132229", 2, True, False),
            "1": ("fed-lr-sched0-1cpr64-bs32-20241022_135427", 1, True, False),
        },
    },
    "partial_participation": {
        # Participation ratios - 512 steps per round
        "0_5": {
            "8": ("fed-pp0_5-8cpr512-bs32-20241026_150027", 8, True, False),
            "4": ("fed-pp0_5-4cpr512-bs32-20241026_173006", 4, True, False),
            "2": ("", 2, True, False),
            "1": ("", 1, True, False),
        },
        "0_25": {
            "8": ("", 8, True, False),
            "4": ("", 4, True, False),
            "2": ("fed-pp0_25-2cpr512-bs32-20241027_174630", 2, True, False),
            "1": ("", 1, True, False),
        },
        "0_25": {
            "8": ("", 8, True, False),
            "4": ("", 4, True, False),
            "2": ("", 2, True, False),
            "1": ("", 1, True, False),
        },
        "0_125": {
            "8": ("fed-pp0_125-8cpr512-bs32-20241026_235833", 8, True, False),
            "4": ("fed-pp0_125-4cpr512-bs32-20241027_030224", 4, True, False),
            "2": ("", 2, True, False),
            "1": ("", 1, True, False),
        },
    },
    "outer_optimizer": {
        # Types of outer optimizers - 512 steps per round
        # partial participation?
        "diloco_0_1": {
            "8": (
                "fed-opt-8cpr512-bs32-20241028_073346",
                8,
                True,
                False,
            ),
            "4": (
                "fed-opt-4cpr512-bs32-20241028_073249",
                4,
                True,
                False,
            ),
            "2": (
                "fed-opt-2cpr512-bs32-20241028_093045",
                2,
                True,
                False,
            ),  # Lorenzo Monitoring on Lambda cluster
            "1": ("", 1, True, False),
        },
        "diloco_0_3": {
            "8": ("", 8, True, False),  # Lorenzo Monitoring on Lambda cluster
            "4": ("fed-opt-4cpr512-bs32-20241027_235049", 4, True, False),
            "2": ("", 2, True, False),  # Lorenzo Monitoring on Lambda cluster
            "1": ("", 1, True, False),
        },
        "diloco_0_5": {
            "8": (
                "fed-opt-8cpr512-bs32-20241027_234049",
                8,
                True,
                False,
            ),
            "4": ("fed-opt-4cpr512-bs32-20241027_210455", 4, True, False),
            "2": ("", 2, True, False),  # Lorenzo Monitoring on Lambda cluster
            "1": ("", 1, True, False),
        },
        "diloco_0_7": {
            "8": (
                "fed-opt-8cpr512-bs32-20241028_154101",
                8,
                True,
                False,
            ),  # Lorenzo Monitoring on Lambda cluster
            "4": ("fed-opt-4cpr512-bs32-20241027_174747", 4, True, False),
            "2": ("", 2, True, False),  # Lorenzo Monitoring on Lambda cluster
            "1": ("", 1, True, False),
        },
    },
    "non_iid_data": {
        # Heterogenous data sources - 512 steps per round (eval on C4 test set)
        # Full participation using 4 data sources each of which is split into 2 for the
        # 8cpr and 4 for the 16cpr.
        "the_pile": {
            "16": ("fed-niid-16cpr512-bs32-20241028_080821", 16, True, False),
            "8": ("fed-niid-8cpr512-bs32-20241028_080956", 8, True, False),
            "4": ("fed-niid-4cpr512-bs32-20241027_221627", 4, True, False),
        },
        "the_pile_partial": {
            "8": ("fed-ppniid-8cpr512-bs32-20241028_153117", 8, True, False),
            "4": ("fed-ppniid-4cpr512-bs32-20241028_144755", 4, True, False),
        },
    },
    # ?
    "reset_opt_states": {},
}

In [9]:
# Server metrics columns
server_metrics_columns = [
    # Train-specific metrics
    "LanguageCrossEntropy",
    "LanguagePerplexity",
    # Val-specific metrics
    "ValLanguageCrossEntropy",
    "ValLanguagePerplexity",
    # Wandb internals
    "_runtime",
    "_step",
    "_timestamp",
    # Client-side bookkeeping
    "client/eval_init_time",
    "client/eval_metrics_collection_time",
    "client/eval_time",
    "client/eval_trainer_closing_time",
    "client/fit_get_parameters_time",
    "client/fit_init_time",
    "client/fit_metrics_collection_time",
    "client/fit_time",
    "client/fit_trainer_closing_time",
    # Client-side metrics
    "client/l2_norm_pseudo_gradient",
    "client_state_acc",
    "distributed_loss",
    # Node-side bookkeeping
    "node_eval_time_s",
    "node_training_time_s",
    # Server-side bookkeeping
    "server/evaluate_round_time",
    "server/evaluate_time",
    "server/first_check_nm_time",
    "server/fit_round_time",
    "server/round_time",
    "server/second_check_nm_time",
    # Server-side computed norms
    "server/l2_norm_fedavg_result",
    "server/l2_norm_model",
    "server/l2_norm_momentum_vector",
    "server/l2_norm_pseudo_gradient",
    # Step
    "step",
    # Worker-side bookkeeping
    "worker/partial_aggregation_time",
]

In [10]:
# Client metrics columns
client_metrics_columns = [
    # Wandb internals
    "_runtime",
    "_step",
    "_timestamp",
    # Activation norms
    "activations/l2_norm/full_model_input",
    "activations/l2_norm/full_model_output",
    # CID
    "client_id",
    # Momentum, gradient and model norms
    "l2_norm/grad/global",
    "l2_norm/moment/global",
    "l2_norm/param/global",
    "l2_norm/update/global",
    # Train loss
    "loss/train/total",
    # LR
    "lr-DecoupledAdamW/group0",
    # Memory metrics
    "memory/alloc_retries",
    "memory/current_active_mem",
    "memory/current_allocated_mem",
    "memory/current_inactive_mem",
    "memory/current_reserved_mem",
    "memory/peak_active_mem",
    "memory/peak_allocated_mem",
    "memory/peak_inactive_mem",
    "memory/peak_reserved_mem",
    "metrics/train/LanguageCrossEntropy",
    "metrics/train/LanguagePerplexity",
    # Step
    "step",
    # Throughput metrics
    "throughput/batches_per_sec",
    "throughput/device/batches_per_sec",
    "throughput/device/flops_per_sec",
    "throughput/device/mfu",
    "throughput/device/samples_per_sec",
    "throughput/device/tokens_per_sec",
    "throughput/flops_per_sec",
    "throughput/samples_per_sec",
    "throughput/tokens_per_sec",
    # Time metrics
    "time/batch",
    "time/batch_in_epoch",
    "time/epoch",
    "time/remaining_estimate",
    "time/sample",
    "time/sample_in_epoch",
    "time/token",
    "time/token_in_epoch",
    "time/total",
    "time/train",
    "time/val",
    # Microbatch size
    "trainer/device_train_microbatch_size",
]

In [11]:
x_lim: tuple[float, float] | None = None
perplexity_y_lim: dict[str, float] | None = {"top": 225, "bottom": 0}
norm_y_lim: dict[str, float] | None = {"bottom": 300, "top": 900}
momentum_y_lim: dict[str, float] | None = {"bottom": 300, "top": 900}
grad_y_lim: dict[str, float] | None = {"bottom": 0, "top": 125}
grad_client_y_lim: dict[str, float] | None = {"bottom": 0, "top": 1.6}
lr_y_lim: dict[str, float] | None = None
act_out_y_lim: dict[str, float] | None = None
act_out_y_lim: dict[str, float] | None = {"bottom": 0, "top": 1.3 * 1e7}
perplexity_legend_kwargs: dict[str, Any] | None = None
l2_gradient_legend_kwargs: dict[str, Any] | None = None

In [12]:
def download_metrics(
    run_id: str, n_clients: int, drop_layers: bool = True, use_server_name: bool = False
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Download all the metrics from given run_id using Wandb API."""
    # Initialize the wandb API
    api = wandb.Api(timeout=100)
    # Get the run object
    server_run = (
        api.run(f"camlsys/pollen-llm/{run_id}_server")
        if use_server_name
        else api.run(f"camlsys/pollen-llm/{run_id}")
    )
    # Get the metrics
    server_metrics = server_run.scan_history()
    # Convert the metrics to a pandas data frame
    server_metrics_df = pd.DataFrame(server_metrics)
    # Drop the layers columns if needed
    if drop_layers:
        layer_columns = [col for col in server_metrics_df.columns if "layer" in col]
        server_metrics_df = server_metrics_df.drop(columns=layer_columns)
    # Add the `step` column based on the index
    server_metrics_df["step"] = server_metrics_df.index
    # Get clients metrics
    clients_metrics_df_list: list[pd.DataFrame] = []

    for i in range(n_clients):
        try:
            # Get the run object

            client_run = (
                api.run(f"camlsys/pollen-llm/{run_id}_client_{i}")
                if (
                    run_id
                    not in {
                        "pile-75M-final-20240605_133225",
                        "partial-75M-20240605_133326",
                    }
                )
                else api.run(f"camlsys/pollen-llm/{run_id}client_{i}")
            )
            # Get the metrics
            client_metrics = client_run.scan_history()
            # Convert the metrics to a pandas data frame
            client_metrics_df = pd.DataFrame(client_metrics)
            # Drop the layers columns if needed
            if drop_layers:
                layer_columns = [
                    col for col in client_metrics_df.columns if "layer" in col
                ]
                client_metrics_df = client_metrics_df.drop(columns=layer_columns)
            # Add the `step` column based on the index
            client_metrics_df["step"] = client_metrics_df._step
            # Add the `client_id` column based on the current client_id
            client_metrics_df["client_id"] = i
            # Append the client metrics to the list
            clients_metrics_df_list.append(client_metrics_df)
        except Exception as e:
            log(
                ERROR,
                f"Client {i} not found for run {run_id}",
                stack_info=True,
                exc_info=e,
            )
    # Concatenate the clients metrics
    clients_metrics_df = pd.concat(clients_metrics_df_list)
    # Return the metrics data frame
    return server_metrics_df, clients_metrics_df

In [13]:
# Model Constants
SERVER_BANDWIDTH = 125  # 125 MBps == 1Gbps
SERVER_BANDWIDTH = 625  # 625 MBps == 5Gbps
SERVER_BANDWIDTH = 1_250  # 10 Gbps
SERVER_BANDWIDTH = 312.5  # 312.5 MBps == 2.5Gbps
SERVER_FLOPS = 5_000_000_000  # 5TFLOPs per second
CHANNELS_THRESHOLD = 100


# Model
class WallTimeModel:
    """Wall time model for the federated learning process."""

    def __init__(
        self,
        num_rounds: int,
        num_clients_per_round: int,
        num_parameters: int,
        local_steps: int,
        local_throughput: float,  # Must be in batches per second
        server_bandwidth: float = SERVER_BANDWIDTH,  # Must be in MBps
        server_flops: float = SERVER_FLOPS,  # Must be in FLOPs per second
        channels_threshold: int = CHANNELS_THRESHOLD,
    ) -> None:
        self.num_rounds = num_rounds
        self.num_clients_per_round = num_clients_per_round
        self.parameters_mbytes = num_parameters
        self.parameters_bytes = 2 * num_parameters
        self.parameters_mbytes = self.parameters_bytes / 10**6
        self.local_steps = local_steps
        self.local_throughput = local_throughput
        self.server_bandwidth = server_bandwidth
        self.server_flops = server_flops
        self.channels_threshold = channels_threshold

    def client_training_time(self) -> float:
        """Compute the time taken for a client to train."""
        # `self.local_throughput` is in batches per second
        # `self.local_steps` is the number of batches trained
        return self.local_steps / self.local_throughput

    def broadcast_model_time(self) -> float:
        """Compute the time taken to broadcast the model."""
        if self.num_clients_per_round == 1:
            return 0.0
        if self.num_clients_per_round > self.channels_threshold:
            return (self.num_clients_per_round * self.parameters_mbytes) / np.sqrt(
                self.server_bandwidth
            )
        else:
            return (
                self.num_clients_per_round * self.parameters_mbytes
            ) / self.server_bandwidth

    def collect_pseudo_gradients_time(self) -> float:
        """Compute the time taken to collect the pseudo gradients."""
        if self.num_clients_per_round == 1:
            return 0.0
        if self.num_clients_per_round > self.channels_threshold:
            return (self.num_clients_per_round * self.parameters_mbytes) / np.sqrt(
                self.server_bandwidth
            )
        else:
            return (
                self.num_clients_per_round * self.parameters_mbytes
            ) / self.server_bandwidth

    def aggregation_time(self) -> float:
        """Compute the time taken to aggregate the pseudo gradients."""
        if self.num_clients_per_round == 1:
            return 0.0
        return 0.0  # (self.num_clients_per_round * self.parameters_mbytes) / self.server_flops

    def allreduce_communication_time(self) -> float:
        """Compute the time taken to perform allreduce communication."""
        if self.num_clients_per_round == 1:
            return 0.0
        return (
            (self.num_clients_per_round - 1) * self.parameters_mbytes
        ) / self.server_bandwidth

    def ring_allreduce_communication_time(self) -> float:
        """Compute the time taken to perform ring allreduce communication."""
        # if self.num_clients_per_round == 1:
        #     return 0.0
        return (
            2
            * self.parameters_mbytes
            * (self.num_clients_per_round - 1)
            / self.num_clients_per_round
        ) / self.server_bandwidth

    def parameter_server_communication_time(self) -> float:
        """Compute the time taken for the parameter server to aggregate the gradients."""
        return (
            self.broadcast_model_time()
            + self.collect_pseudo_gradients_time()
            + self.aggregation_time()
        )

    def allreduce_server_time(self) -> float:
        """Compute the time taken for the parameter server to perform allreduce."""
        return self.allreduce_communication_time() + self.aggregation_time()

    def ring_allreduce_server_time(self) -> float:
        """Compute the time taken for the parameter server to perform ring allreduce."""
        return self.ring_allreduce_communication_time() + self.aggregation_time()

    def round_completion_time_ps(self) -> float:
        """Compute the time taken to complete a round using parameter server."""
        return self.client_training_time() + self.parameter_server_communication_time()

    def round_completion_time_ar(self) -> float:
        """Compute the time taken to complete a round using allreduce."""
        return self.client_training_time() + self.allreduce_server_time()

    def round_completion_time_rar(self) -> float:
        """Compute the time taken to complete a round using ring allreduce."""
        return self.client_training_time() + self.ring_allreduce_server_time()

    def total_wall_time_ps(self) -> float:
        """Compute the total wall time using parameter server."""
        return self.num_rounds * self.round_completion_time_ps()

    def total_wall_time_ar(self) -> float:
        """Compute the total wall time using allreduce."""
        return self.num_rounds * self.round_completion_time_ar()

    def total_wall_time_rar(self) -> float:
        """Compute the total wall time using ring allreduce."""
        return self.num_rounds * self.round_completion_time_rar()

In [14]:
# Centralized baseline (bs256) - PPl = 42
cen_bs256_ppl42 = WallTimeModel(
    num_rounds=3500,
    num_clients_per_round=8,
    num_parameters=125_000_000,
    local_steps=1,
    local_throughput=2,  # 2 batches per second (empirical on 1xH100)
)
# Centralized baseline (bs256) - PPl = 35
cen_bs256_ppl35 = WallTimeModel(
    num_rounds=5950,
    num_clients_per_round=8,
    num_parameters=125_000_000,
    local_steps=1,
    local_throughput=2,  # 2 batches per second (empirical on 1xH100)
)

In [15]:
def find_federated_round_for_perplexity(
    client_metrics_df: pd.DataFrame,
    perplexity_value: float | None,
    rolling_window: int = 100,
) -> tuple[int, float]:
    """
    Find the federated round for a given perplexity value.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing the metrics with columns 'steps' and 'metrics/train/LanguagePerplexity'.
    perplexity_value : float
        The perplexity value to find the federated round for.
    rolling_window : int, optional
        The window size for smoothing the perplexity series, by default 100.

    Returns
    -------
    int
        The federated round corresponding to the given perplexity value.
    """
    # Get columns
    exclude_columns = client_metrics_df.columns
    # Remove columns of interest
    exclude_columns = [
        col
        for col in exclude_columns
        if col not in {"step", "metrics/train/LanguagePerplexity"}
    ]
    aggregated_df = client_metrics_df.drop(columns=exclude_columns)
    # Remove inf
    aggregated_df = aggregated_df.replace([np.inf, -np.inf], np.nan)
    # Substitute non-numeric values with NaN
    aggregated_df = aggregated_df.apply(pd.to_numeric, errors="coerce")
    # Aggregate by step and remove nans
    aggregated_df = aggregated_df.groupby("step").mean().reset_index()
    # Fill NaNs by interpolation
    aggregated_df = aggregated_df.interpolate()
    # Extract `metrics/train/LanguagePerplexity` series against steps
    steps: pd.Series[int] = aggregated_df["step"]
    perplexity_series = aggregated_df["metrics/train/LanguagePerplexity"]

    # Smooth the line using a rolling window of 100 steps
    smoothed_perplexity = perplexity_series.rolling(window=rolling_window).mean()
    # smoothed_perplexity = perplexity_series.rolling(window=rolling_window).min()
    # log(INFO, f"Smoothed perplexity: {smoothed_perplexity}")

    # Find the x-coordinate of a given perplexity value
    if perplexity_value is None:
        perplexity_value = smoothed_perplexity.min()
    assert perplexity_value is not None
    closest_index = (smoothed_perplexity - perplexity_value).abs().idxmin()
    x_coordinate = steps[closest_index]
    # log(INFO, f"Closest index: {closest_index}, x-coordinate: {x_coordinate}")

    # Return the step and the perplexity value
    return x_coordinate, perplexity_value

In [16]:
# Load the WandB data
experiment_type = "full_participation"
for i, min_ppl in enumerate(reversed([35, 42])):
    fig = plt.figure(figsize=fig_size2, dpi=dpi)
    for j, experiment_hp in enumerate(run_id_dict[experiment_type]):
        y_coordinates: list[int] = []
        x_coordinates: list[float] = []
        for conf in sorted(
            [int(x) for x in run_id_dict[experiment_type][experiment_hp]]
        ):
            run_uuid = run_id_dict[experiment_type][experiment_hp][str(conf)][0]
            try:
                server_metrics_df = pd.read_pickle(f"server_{run_uuid}.pkl")
                client_metrics_df = pd.read_pickle(f"clients_{run_uuid}.pkl")
            except FileNotFoundError:
                server_metrics_df, client_metrics_df = download_metrics(
                    *run_id_dict[experiment_type][experiment_hp][str(conf)]
                )
                # Dump the dataframes to pickle files
                server_metrics_df.to_pickle(f"server_{run_uuid}.pkl")
                client_metrics_df.to_pickle(f"clients_{run_uuid}.pkl")
            steps_to_min, new_min_ppl = find_federated_round_for_perplexity(
                client_metrics_df=client_metrics_df,
                perplexity_value=min_ppl,
                rolling_window=100,
            )
            # Initialize the model
            n_local_steps = int(experiment_hp.replace("_steps", ""))
            # Assuming conf, n_local_steps, and min_ppl are already defined
            tokens = steps_to_min * 32 * 2028 * conf

            if tokens >= 1_000_000_000:
                tokens_str = f"{tokens / 1_000_000_000:.2f}B"
            elif tokens >= 1_000_000:
                tokens_str = f"{tokens / 1_000_000:.2f}M"
            else:
                tokens_str = str(tokens)
            log(
                INFO,
                "Experiment with %s clients per round trained on %s tokens in %s steps to achieve %s perplexity",
                conf,
                tokens_str,
                steps_to_min,
                min_ppl,
            )
            n_rounds = steps_to_min // n_local_steps
            model = WallTimeModel(
                num_rounds=n_rounds,
                num_clients_per_round=int(conf),
                num_parameters=125_000_000,
                local_steps=n_local_steps,
                local_throughput=2,  # 2 batches per second
            )

            # Calculate total wall time for different approaches
            total_wall_time_ps = model.total_wall_time_ps()
            total_wall_time_ar = model.total_wall_time_ar()
            total_wall_time_rar = model.total_wall_time_rar()

            y_coordinates.append(int(conf))
            log(
                INFO,
                "n_local_steps=%s, num_clients_per_round=%s, n_rounds=%s, min_ppl=%s",
                n_local_steps,
                int(conf),
                n_rounds,
                min_ppl,
            )
            if int(conf) == 1:
                x_coordinates.append(steps_to_min / model.local_throughput)
            else:
                x_coordinates.append(total_wall_time_rar)
        # Plot the y_coordinates against the x_coordinates
        plt.plot(
            x_coordinates,
            [y * 32 for y in y_coordinates],
            marker=".",
            label=f"{experiment_hp.replace('_', ' ')}",
            color=color_palette[j],
            linestyle=line_styles[i],
        )

    plt.xlabel("Wall Time (s)")
    plt.legend(
        loc="upper right",
        fontsize="small",
        framealpha=0.5,
        ncol=1,
        title=f"Perplexity={min_ppl}",
    )
    plt.xscale("log")
    plt.yscale("log")
    # Change the minor grid to have a smaller linewidth
    plt.grid(axis="both", which="minor", lw=0.5)
    plt.grid(axis="both", which="major")

    plt.ylabel(r"Global Batch Size $B_g$")
    plt.yticks([32, 64, 128, 256, 512], labels=["32", "64", "128", "256", "512"])

    if i == 0:
        # plt.ylabel(r"Global Batch Size $B_g$")
        # Change the y ticks to report only the values 1, 2, 4, 8, 16
        # plt.yticks([32, 64, 128, 256, 512], labels=["32", "64", "128", "256", "512"])
        # Change the x tciks
        plt.xticks(
            [4_000, 4_100, 5000, 6000],
            labels=[
                "",
                r"$4.1_{\times10^3}$",
                r"$5_{\times10^3}$",
                r"$6_{\times10^3}$",
            ],
        )
    else:
        # Remove the y ticks
        # plt.yticks([32, 64, 128, 256, 512], labels=["", "", "", "", ""])
        # Change the x tciks
        plt.xticks(
            [8000, 9000, 10000, 12000],
            labels=[
                r"$8_{\times10^3}$",
                r"$9_{\times10^3}$",
                r"$1_{\times10^4}$",
                r"$1.2_{\times10^4}$",
            ],
        )
    plt.savefig(
        f"wall_time_{experiment_type}_ppl{min_ppl}.pdf", bbox_inches="tight", dpi=dpi
    )
    plt.show()

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:wandb: Appending key for api.wandb.ai to your netrc file: /Users/iacobalexandru/.netrc
INFO :      Closest index: 24669, x-coordinate: 24980
INFO :      Min PPL for conf 1: 36 at step 24980
INFO :      Total Wall Time for 1 (Parameter Server): 288.6000000195
INFO :   

In [ ]:
# Load the WandB data
experiment_type = "outer_optimizer"
for i, min_ppl in enumerate(reversed([35, 42])):
    fig = plt.figure(figsize=fig_size, dpi=dpi)
    for j, experiment_hp in enumerate(run_id_dict[experiment_type]):
        if experiment_hp != "diloco_0_1":
            continue
        y_coordinates: list[int] = []
        x_coordinates: list[float] = []
        for conf in sorted(
            [int(x) for x in run_id_dict[experiment_type][experiment_hp]]
        ):
            run_uuid = run_id_dict[experiment_type][experiment_hp][str(conf)][0]
            if run_uuid == "":
                continue
            try:
                server_metrics_df = pd.read_pickle(f"server_{run_uuid}.pkl")
                client_metrics_df = pd.read_pickle(f"clients_{run_uuid}.pkl")
            except FileNotFoundError:
                server_metrics_df, client_metrics_df = download_metrics(
                    *run_id_dict[experiment_type][experiment_hp][str(conf)]
                )
                # Dump the dataframes to pickle files
                server_metrics_df.to_pickle(f"server_{run_uuid}.pkl")
                client_metrics_df.to_pickle(f"clients_{run_uuid}.pkl")
            steps_to_min, new_min_ppl = find_federated_round_for_perplexity(
                client_metrics_df=client_metrics_df,
                perplexity_value=min_ppl,
                rolling_window=100,
            )
            min_ppl = min(min_ppl, new_min_ppl) if min_ppl is not None else new_min_ppl
            # Initialize the model
            n_local_steps = 512
            n_rounds = steps_to_min // n_local_steps
            model = WallTimeModel(
                num_rounds=n_rounds,
                num_clients_per_round=int(conf),
                num_parameters=125_000_000,
                local_steps=n_local_steps,
                local_throughput=2,  # 2 batches per second
            )

            # Calculate total wall time for different approaches
            total_wall_time_ps = model.total_wall_time_ps()
            total_wall_time_ar = model.total_wall_time_ar()
            total_wall_time_rar = model.total_wall_time_rar()

            y_coordinates.append(int(conf))
            x_coordinates.append(total_wall_time_rar)
        # Plot the y_coordinates against the x_coordinates
        plt.plot(
            x_coordinates,
            [y * 32 for y in y_coordinates],
            marker=".",
            label=r"DiLoCo $(\eta_s=0.1)$",
            color=color_palette[j],
            linestyle=line_styles[i],
        )
    for j, experiment_hp in enumerate(run_id_dict["full_participation"]):
        if experiment_hp != "512_steps":
            continue
        y_coordinates: list[int] = []
        # x_coordinates: list[int] = []
        x_coordinates: list[float] = []
        for conf in sorted(
            [int(x) for x in run_id_dict["full_participation"][experiment_hp]]
        ):
            if conf not in {2, 4, 8}:
                continue
            run_uuid = run_id_dict["full_participation"][experiment_hp][str(conf)][0]
            if run_uuid == "":
                continue
            try:
                server_metrics_df = pd.read_pickle(f"server_{run_uuid}.pkl")
                client_metrics_df = pd.read_pickle(f"clients_{run_uuid}.pkl")
            except FileNotFoundError:
                server_metrics_df, client_metrics_df = download_metrics(
                    *run_id_dict["full_participation"][experiment_hp][str(conf)]
                )
                # Dump the dataframes to pickle files
                server_metrics_df.to_pickle(f"server_{run_uuid}.pkl")
                client_metrics_df.to_pickle(f"clients_{run_uuid}.pkl")
            steps_to_min, new_min_ppl = find_federated_round_for_perplexity(
                client_metrics_df=client_metrics_df,
                perplexity_value=min_ppl,
                rolling_window=100,
            )
            min_ppl = min(min_ppl, new_min_ppl) if min_ppl is not None else new_min_ppl
            # Initialize the model
            n_local_steps = 512
            n_rounds = steps_to_min // n_local_steps
            model = WallTimeModel(
                num_rounds=n_rounds,
                num_clients_per_round=int(conf),
                num_parameters=125_000_000,
                local_steps=n_local_steps,
                local_throughput=2,  # 2 batches per second
            )

            # Calculate total wall time for different approaches
            total_wall_time_ps = model.total_wall_time_ps()
            total_wall_time_ar = model.total_wall_time_ar()
            total_wall_time_rar = model.total_wall_time_rar()

            y_coordinates.append(int(conf))
            x_coordinates.append(total_wall_time_rar)
        # Plot the y_coordinates against the x_coordinates
        plt.plot(
            x_coordinates,
            [y * 32 for y in y_coordinates],
            marker=".",
            label="SystemX",
            color=color_palette[j],
            linestyle=line_styles[i],
        )
    plt.xlabel("Wall Time")
    if i == 0:
        plt.ylabel(r"Global Batch Size $B_g$")
    plt.xscale("log")
    plt.yscale("log")
    # Change the minor grid to have a smaller linewidth
    plt.grid(axis="both", which="minor", lw=0.5)
    plt.grid(axis="both", which="major")
    # Get the displayed xticks labels
    xticks_labels = plt.xticks()[0]
    log(INFO, xticks_labels)
    # Change the y ticks to report only the values 1, 2, 4, 8, 16
    plt.yticks([60, 64, 128, 200, 256], labels=["", "64", "128", "", "256"])
    # Change the x ticks
    if i == 0:
        plt.xticks(
            [5000, 6000, 7000, 8000, 9000, 10000],
            labels=[
                r"$5_{\times10^3}$",
                "",
                r"$6_{\times10^3}$",
                "",
                "",
                r"$1_{\times10^4}$",
            ],
        )
    else:
        # Remove the y ticks
        plt.yticks([60, 64, 128, 200, 256], labels=["", "", "", "", ""])
        plt.xticks(
            [10000, 12000, 14000, 15000, 16000, 18000, 20000],
            labels=[
                r"$1_{\times10^4}$",
                "",
                "",
                r"$1.5_{\times10^4}$",
                "",
                "",
                r"$2_{\times10^4}$",
            ],
        )
    plt.legend(
        loc="center",
        fontsize="small",
        framealpha=0.5,
        ncol=1,
        title=f"Perplexity={min_ppl}",
    )
    plt.savefig(
        f"wall_time_{experiment_type}_ppl{min_ppl}.pdf", bbox_inches="tight", dpi=dpi
    )
    plt.show()

In [ ]:
import pandas as pd

# Load the WandB data
experiment_type = "outer_optimizer"
table_data = []

for i, min_ppl in enumerate(reversed([35, 42])):
    for j, experiment_hp in enumerate(run_id_dict[experiment_type]):
        if experiment_hp != "diloco_0_1":
            continue
        for conf in sorted(
            [int(x) for x in run_id_dict[experiment_type][experiment_hp]]
        ):
            run_uuid = run_id_dict[experiment_type][experiment_hp][str(conf)][0]
            if run_uuid == "":
                continue
            try:
                server_metrics_df = pd.read_pickle(f"server_{run_uuid}.pkl")
                client_metrics_df = pd.read_pickle(f"clients_{run_uuid}.pkl")
            except FileNotFoundError:
                server_metrics_df, client_metrics_df = download_metrics(
                    *run_id_dict[experiment_type][experiment_hp][str(conf)]
                )
                # Dump the dataframes to pickle files
                server_metrics_df.to_pickle(f"server_{run_uuid}.pkl")
                client_metrics_df.to_pickle(f"clients_{run_uuid}.pkl")
            steps_to_min, new_min_ppl = find_federated_round_for_perplexity(
                client_metrics_df=client_metrics_df,
                perplexity_value=min_ppl,
                rolling_window=100,
            )
            min_ppl = min(min_ppl, new_min_ppl) if min_ppl is not None else new_min_ppl
            # Initialize the model
            n_local_steps = 512
            n_rounds = steps_to_min // n_local_steps
            model = WallTimeModel(
                num_rounds=n_rounds,
                num_clients_per_round=int(conf),
                num_parameters=125_000_000,
                local_steps=n_local_steps,
                local_throughput=2,  # 2 batches per second
            )

            # Calculate total wall time for different approaches
            total_wall_time_ps = model.total_wall_time_ps()
            total_wall_time_ar = model.total_wall_time_ar()
            total_wall_time_rar = model.total_wall_time_rar()

            table_data.append(
                {
                    "Method": "DiLoCo $(\eta_s=0.1)$",
                    "Clients": conf,
                    "Total Wall Time RAR": total_wall_time_rar,
                    "Target Perplexity": min_ppl,
                }
            )

    for j, experiment_hp in enumerate(run_id_dict["full_participation"]):
        if experiment_hp != "512_steps":
            continue
        for conf in sorted(
            [int(x) for x in run_id_dict["full_participation"][experiment_hp]]
        ):
            if conf not in {2, 4, 8}:
                continue
            run_uuid = run_id_dict["full_participation"][experiment_hp][str(conf)][0]
            if run_uuid == "":
                continue
            try:
                server_metrics_df = pd.read_pickle(f"server_{run_uuid}.pkl")
                client_metrics_df = pd.read_pickle(f"clients_{run_uuid}.pkl")
            except FileNotFoundError:
                server_metrics_df, client_metrics_df = download_metrics(
                    *run_id_dict["full_participation"][experiment_hp][str(conf)]
                )
                # Dump the dataframes to pickle files
                server_metrics_df.to_pickle(f"server_{run_uuid}.pkl")
                client_metrics_df.to_pickle(f"clients_{run_uuid}.pkl")
            steps_to_min, new_min_ppl = find_federated_round_for_perplexity(
                client_metrics_df=client_metrics_df,
                perplexity_value=min_ppl,
                rolling_window=100,
            )
            min_ppl = min(min_ppl, new_min_ppl) if min_ppl is not None else new_min_ppl
            # Initialize the model
            n_local_steps = 512
            n_rounds = steps_to_min // n_local_steps
            model = WallTimeModel(
                num_rounds=n_rounds,
                num_clients_per_round=int(conf),
                num_parameters=125_000_000,
                local_steps=n_local_steps,
                local_throughput=2,  # 2 batches per second
            )

            # Calculate total wall time for different approaches
            total_wall_time_ps = model.total_wall_time_ps()
            total_wall_time_ar = model.total_wall_time_ar()
            total_wall_time_rar = model.total_wall_time_rar()

            table_data.append(
                {
                    "Method": "SystemX",
                    "Clients": conf,
                    "Total Wall Time RAR": total_wall_time_rar,
                    "Target Perplexity": min_ppl,
                }
            )

# Convert the table data to a DataFrame and save it as a CSV file
table_df = pd.DataFrame(table_data)
# table_df.to_csv("wall_time_table.csv", index=False)
log(INFO, "Table: %s", table_df)

In [ ]:
# Load the WandB data
experiment_type = "outer_optimizer"
fig = plt.figure(figsize=fig_size1, dpi=dpi)
for j, experiment_hp in enumerate(run_id_dict[experiment_type]):
    y_coordinates: list[int] = []
    x_coordinates: list[float] = []
    for conf in sorted([int(x) for x in run_id_dict[experiment_type][experiment_hp]]):
        run_uuid = run_id_dict[experiment_type][experiment_hp][str(conf)][0]
        if conf != 4:
            continue
        if run_uuid == "":
            continue
        try:
            client_metrics_df = pd.read_pickle(f"clients_{run_uuid}.pkl")
        except FileNotFoundError:
            _server_metrics_df, client_metrics_df = download_metrics(
                *run_id_dict[experiment_type][experiment_hp][str(conf)]
            )
            # Dump the dataframes to pickle files
            client_metrics_df.to_pickle(f"clients_{run_uuid}.pkl")
        steps_series, ppl_series = get_perplexity_series(
            client_metrics_df, rolling_window=100
        )
        rounds_series = steps_series // 512
        server_learning_rate = float(
            experiment_hp.replace("diloco_", "").replace("_", ".")
        )
        # Plot the y_coordinates against the x_coordinates
        plt.plot(
            rounds_series,
            ppl_series,
            label=rf"DiLoCo $(\eta_s={server_learning_rate})$",
            color=color_palette[j],
            linestyle=line_styles[0],
        )
run_uuid = run_id_dict["full_participation"]["512_steps"]["4"][0]
try:
    client_metrics_df = pd.read_pickle(f"clients_{run_uuid}.pkl")
except FileNotFoundError:
    _server_metrics_df, client_metrics_df = download_metrics(
        *run_id_dict["full_participation"]["512_steps"]["4"]
    )
    # Dump the dataframes to pickle files
    client_metrics_df.to_pickle(f"clients_{run_uuid}.pkl")
steps_series, ppl_series = get_perplexity_series(client_metrics_df, rolling_window=100)
rounds_series = steps_series // 512
# Plot the y_coordinates against the x_coordinates
plt.plot(
    rounds_series,
    ppl_series,
    label="SystemX",
    color=color_palette[j + 1],
    linestyle=line_styles[0],
)
plt.xlabel("Federated Rounds")
plt.ylabel("Perplexity")
# Change the minor grid to have a smaller linewidth
plt.grid(axis="both", which="minor", lw=0.5)
plt.grid(axis="both", which="major")
# Set y-axis limits
plt.ylim(20, 100)
plt.legend()
plt.savefig("diloco_ppl_4cpr.pdf", bbox_inches="tight", dpi=dpi)
plt.show()

In [ ]:
# Load the WandB data
experiment_type = "non_iid_data"
experiment_hp = "the_pile_partial"
fig = plt.figure(figsize=fig_size1, dpi=dpi)
y_coordinates: list[int] = []
# x_coordinates: list[int] = []
x_coordinates: list[float] = []
j = 0
for j, conf in enumerate(
    sorted([int(x) for x in run_id_dict[experiment_type][experiment_hp]])
):
    run_uuid = run_id_dict[experiment_type][experiment_hp][str(conf)][0]
    if run_uuid == "":
        continue
    try:
        client_metrics_df = pd.read_pickle(f"clients_{run_uuid}.pkl")
    except FileNotFoundError:
        _server_metrics_df, client_metrics_df = download_metrics(
            *run_id_dict[experiment_type][experiment_hp][str(conf)]
        )
        # Dump the dataframes to pickle files
        client_metrics_df.to_pickle(f"clients_{run_uuid}.pkl")
    steps_series, ppl_series = get_perplexity_series(
        client_metrics_df, rolling_window=100
    )
    rounds_series = steps_series // 512
    # Plot the y_coordinates against the x_coordinates
    plt.plot(
        rounds_series,
        ppl_series,
        label=f"non-IID - {conf} clients",
        color=color_palette[j],
        linestyle=line_styles[0],
    )
run_uuid = run_id_dict["full_participation"]["512_steps"]["4"][0]
try:
    client_metrics_df = pd.read_pickle(f"clients_{run_uuid}.pkl")
except FileNotFoundError:
    _server_metrics_df, client_metrics_df = download_metrics(
        *run_id_dict["full_participation"]["512_steps"]["4"]
    )
    # Dump the dataframes to pickle files
    client_metrics_df.to_pickle(f"clients_{run_uuid}.pkl")
steps_series, ppl_series = get_perplexity_series(client_metrics_df, rolling_window=100)
rounds_series = steps_series // 512
# Plot the y_coordinates against the x_coordinates
plt.plot(
    rounds_series,
    ppl_series,
    label="IID - 4 clients",
    color=color_palette[j + 1],
    linestyle=line_styles[0],
)
run_uuid = run_id_dict["full_participation"]["512_steps"]["8"][0]
try:
    client_metrics_df = pd.read_pickle(f"clients_{run_uuid}.pkl")
except FileNotFoundError:
    _server_metrics_df, client_metrics_df = download_metrics(
        *run_id_dict["full_participation"]["512_steps"]["8"]
    )
    # Dump the dataframes to pickle files
    client_metrics_df.to_pickle(f"clients_{run_uuid}.pkl")
steps_series, ppl_series = get_perplexity_series(client_metrics_df, rolling_window=100)
rounds_series = steps_series // 512
# Plot the y_coordinates against the x_coordinates
plt.plot(
    rounds_series,
    ppl_series,
    label="IID - 8 clients",
    color=color_palette[j + 2],
    linestyle=line_styles[0],
)
plt.xlabel("Federated Rounds")
plt.ylabel("Perplexity")
# Change the minor grid to have a smaller linewidth
plt.grid(axis="both", which="minor", lw=0.5)
plt.grid(axis="both", which="major")
# Set y-axis limits
plt.ylim(20, 50)
plt.legend()
plt.savefig("hetero_pp_ppl.pdf", bbox_inches="tight", dpi=dpi)
plt.show()

In [ ]:
# Load the WandB data
experiment_type = "non_iid_data"
experiment_hp = "the_pile"
fig = plt.figure(figsize=fig_size1, dpi=dpi)
y_coordinates: list[int] = []
# x_coordinates: list[int] = []
x_coordinates: list[float] = []
j = 0
for j, conf in enumerate(
    sorted([int(x) for x in run_id_dict[experiment_type][experiment_hp]])
):
    run_uuid = run_id_dict[experiment_type][experiment_hp][str(conf)][0]
    if run_uuid == "":
        continue
    try:
        client_metrics_df = pd.read_pickle(f"clients_{run_uuid}.pkl")
    except FileNotFoundError:
        _server_metrics_df, client_metrics_df = download_metrics(
            *run_id_dict[experiment_type][experiment_hp][str(conf)]
        )
        # Dump the dataframes to pickle files
        client_metrics_df.to_pickle(f"clients_{run_uuid}.pkl")
    steps_series, ppl_series = get_perplexity_series(
        client_metrics_df, rolling_window=100
    )
    rounds_series = steps_series // 512
    # Plot the y_coordinates against the x_coordinates
    plt.plot(
        rounds_series,
        ppl_series,
        label=f"non-IID - {conf} clients",
        color=color_palette[j],
        linestyle=line_styles[0],
    )
run_uuid = run_id_dict["full_participation"]["512_steps"]["16"][0]
try:
    client_metrics_df = pd.read_pickle(f"clients_{run_uuid}.pkl")
except FileNotFoundError:
    _server_metrics_df, client_metrics_df = download_metrics(
        *run_id_dict["full_participation"]["512_steps"]["16"]
    )
    # Dump the dataframes to pickle files
    client_metrics_df.to_pickle(f"clients_{run_uuid}.pkl")
steps_series, ppl_series = get_perplexity_series(client_metrics_df, rolling_window=100)
rounds_series = steps_series // 512
# Plot the y_coordinates against the x_coordinates
plt.plot(
    rounds_series,
    ppl_series,
    label="IID - 16 clients",
    color=color_palette[j + 1],
    linestyle=line_styles[0],
)
plt.xlabel("Federated Rounds")
plt.ylabel("Perplexity")
# Change the minor grid to have a smaller linewidth
plt.grid(axis="both", which="minor", lw=0.5)
plt.grid(axis="both", which="major")
# Set y-axis limits
plt.ylim(20, 50)
plt.legend()
plt.savefig("hetero_fp_ppl.pdf", bbox_inches="tight", dpi=dpi)
plt.show()

In [ ]:
# Load the WandB data
experiment_type = "full_participation"
experiment_hp = "512_steps"
min_ppl = 35  # Cherry-picked value
bar_width = 0.3  # Width of each bar
fig = plt.figure(figsize=fig_size1, dpi=dpi)

# Set grid lines with lower zorder to ensure they are in the background
plt.grid(axis="y", which="minor", lw=0.5, zorder=0)
plt.grid(axis="y", which="major", zorder=0)
y_coordinates: list[int] = []
x_coordinates: list[float] = []
conf_values = sorted([int(x) for x in run_id_dict[experiment_type][experiment_hp]])
j = 0
for i, conf in enumerate(conf_values):
    if conf == 1:
        continue
    run_uuid = run_id_dict[experiment_type][experiment_hp][str(conf)][0]
    try:
        server_metrics_df = pd.read_pickle(f"server_{run_uuid}.pkl")
        client_metrics_df = pd.read_pickle(f"clients_{run_uuid}.pkl")
    except FileNotFoundError:
        server_metrics_df, client_metrics_df = download_metrics(
            *run_id_dict[experiment_type][experiment_hp][str(conf)]
        )
        # Dump the dataframes to pickle files
        server_metrics_df.to_pickle(f"server_{run_uuid}.pkl")
        client_metrics_df.to_pickle(f"clients_{run_uuid}.pkl")
    steps_to_min, new_min_ppl = find_federated_round_for_perplexity(
        client_metrics_df=client_metrics_df,
        perplexity_value=min_ppl,
        rolling_window=100,
    )
    # Initialize the model
    n_local_steps = int(experiment_hp.replace("_steps", ""))
    n_rounds = steps_to_min // n_local_steps
    model = WallTimeModel(
        num_rounds=n_rounds,
        num_clients_per_round=int(conf),
        num_parameters=125_000_000,
        local_steps=n_local_steps,
        local_throughput=2,  # 2 batches per second
    )

    # Calculate total wall time for different approaches
    round_compute_time = model.num_rounds * model.client_training_time()
    rar_rounds_comms_time = model.num_rounds * model.ring_allreduce_communication_time()
    ar_rounds_comms_time = model.num_rounds * model.allreduce_communication_time()
    ps_rounds_comms_time = (
        model.num_rounds * model.parameter_server_communication_time()
    )

    # Construct adjacent bars for each configuration
    plt.bar(
        (i + 1) - bar_width,
        round_compute_time,
        width=bar_width,
        # color="gray",
        color=color_palette[-1],
        edgecolor="black",
        label="LC" if j == 0 else "",
        zorder=3,
    )
    j += 1
    plt.bar(
        (i + 1) - bar_width,
        rar_rounds_comms_time,
        bottom=round_compute_time,
        width=bar_width,
        label="RAR" if i == 1 else "",
        # color="lightgray",
        color=color_palette[-2],
        edgecolor="black",
        zorder=3,
        hatch=patterns[6],
    )
    plt.bar(
        (i + 1),
        round_compute_time,
        width=bar_width,
        # color="gray",
        color=color_palette[-1],
        edgecolor="black",
        zorder=3,
    )
    plt.bar(
        (i + 1),
        ar_rounds_comms_time,
        bottom=round_compute_time,
        width=bar_width,
        label="AR" if i == 1 else "",
        # color="lightgray",
        color=color_palette[-3],
        edgecolor="black",
        zorder=3,
        hatch=patterns[7],
    )
    plt.bar(
        (i + 1) + bar_width,
        round_compute_time,
        width=bar_width,
        # color="gray",
        color=color_palette[-1],
        edgecolor="black",
        zorder=3,
    )
    plt.bar(
        (i + 1) + bar_width,
        ps_rounds_comms_time,
        bottom=round_compute_time,
        width=bar_width,
        label="PS" if i == 1 else "",
        # color="lightgray",
        color=color_palette[-4],
        edgecolor="black",
        zorder=3,
        hatch=patterns[8],
    )

    # Add percentage text on top of the bars
    plt.text(
        (i + 1) - bar_width,
        round_compute_time + rar_rounds_comms_time + 100,
        f"{(rar_rounds_comms_time / (round_compute_time + rar_rounds_comms_time)) * 100:.1f}%",
        ha="center",
        va="bottom",
        fontsize=12,
        zorder=4,
    )
    plt.text(
        (i + 1),
        round_compute_time + ar_rounds_comms_time + 100,
        f"{(ar_rounds_comms_time / (round_compute_time + ar_rounds_comms_time)) * 100:.1f}%",
        ha="center",
        va="bottom",
        fontsize=12,
        zorder=4,
    )
    plt.text(
        (i + 1) + bar_width,
        round_compute_time + ps_rounds_comms_time + 100,
        f"{(ps_rounds_comms_time / (round_compute_time + ps_rounds_comms_time)) * 100:.1f}%",
        ha="center",
        va="bottom",
        fontsize=12,
        zorder=4,
    )

# Customize the legend to assign the color to the number of steps and the line styles to the perplexity value
plt.legend(
    loc="upper left",
    bbox_to_anchor=(0.05, 0.8),  # Adjust this tuple to move the legend
    fontsize="small",
    title_fontsize="medium",
    fancybox=True,
    framealpha=0.8,
    ncol=1,
)
plt.yscale("log")
plt.xticks(
    [2, 3, 4, 5],
    labels=["2", "4", "8", "16"],
)
plt.xlabel("Number of Clients")
plt.ylabel("Wall Time (s)")
plt.ylim(8_000, 10_200)
plt.yticks(
    [8_000, 9_000, 10_000, 10_500],
    labels=[
        r"$8_{\times10^3}$",
        r"$9_{\times10^3}$",
        r"$1_{\times10^4}$",
        "",
    ],
)
plt.savefig("wall_time_split_bar_plot_512steps.pdf", bbox_inches="tight", dpi=dpi)
plt.show()

In [ ]:
# Load the WandB data
experiment_type = "full_participation"
experiment_hp = "128_steps"
min_ppl = 35  # Cherry-picked value
bar_width = 0.3  # Width of each bar
fig = plt.figure(figsize=fig_size1, dpi=dpi)

# Set grid lines with lower zorder to ensure they are in the background
plt.grid(axis="y", which="minor", lw=0.5, zorder=0)
plt.grid(axis="y", which="major", zorder=0)
y_coordinates: list[int] = []
x_coordinates: list[float] = []
conf_values = sorted([int(x) for x in run_id_dict[experiment_type][experiment_hp]])
j = 0
for i, conf in enumerate(conf_values):
    if conf == 1:
        continue
    run_uuid = run_id_dict[experiment_type][experiment_hp][str(conf)][0]
    try:
        server_metrics_df = pd.read_pickle(f"server_{run_uuid}.pkl")
        client_metrics_df = pd.read_pickle(f"clients_{run_uuid}.pkl")
    except FileNotFoundError:
        server_metrics_df, client_metrics_df = download_metrics(
            *run_id_dict[experiment_type][experiment_hp][str(conf)]
        )
        # Dump the dataframes to pickle files
        server_metrics_df.to_pickle(f"server_{run_uuid}.pkl")
        client_metrics_df.to_pickle(f"clients_{run_uuid}.pkl")
    steps_to_min, new_min_ppl = find_federated_round_for_perplexity(
        client_metrics_df=client_metrics_df,
        perplexity_value=min_ppl,
        rolling_window=100,
    )
    # Initialize the model
    n_local_steps = int(experiment_hp.replace("_steps", ""))
    n_rounds = steps_to_min // n_local_steps
    model = WallTimeModel(
        num_rounds=n_rounds,
        num_clients_per_round=int(conf),
        num_parameters=125_000_000,
        local_steps=n_local_steps,
        local_throughput=2,  # 2 batches per second
    )

    # Calculate total wall time for different approaches
    round_compute_time = model.num_rounds * model.client_training_time()
    rar_rounds_comms_time = model.num_rounds * model.ring_allreduce_communication_time()
    ar_rounds_comms_time = model.num_rounds * model.allreduce_communication_time()
    ps_rounds_comms_time = (
        model.num_rounds * model.parameter_server_communication_time()
    )

    # Construct adjacent bars for each configuration
    plt.bar(
        (i + 1) - bar_width,
        round_compute_time,
        width=bar_width,
        # color="gray",
        color=color_palette[-1],
        edgecolor="black",
        label="LC" if j == 0 else "",
        zorder=3,
    )
    j += 1
    plt.bar(
        (i + 1) - bar_width,
        rar_rounds_comms_time,
        bottom=round_compute_time,
        width=bar_width,
        label="RAR" if i == 1 else "",
        # color="lightgray",
        color=color_palette[-2],
        edgecolor="black",
        zorder=3,
        hatch=patterns[6],
    )
    plt.bar(
        (i + 1),
        round_compute_time,
        width=bar_width,
        # color="gray",
        color=color_palette[-1],
        edgecolor="black",
        zorder=3,
    )
    plt.bar(
        (i + 1),
        ar_rounds_comms_time,
        bottom=round_compute_time,
        width=bar_width,
        label="AR" if i == 1 else "",
        # color="lightgray",
        color=color_palette[-3],
        edgecolor="black",
        zorder=3,
        hatch=patterns[7],
    )
    plt.bar(
        (i + 1) + bar_width,
        round_compute_time,
        width=bar_width,
        # color="gray",
        color=color_palette[-1],
        edgecolor="black",
        zorder=3,
    )
    plt.bar(
        (i + 1) + bar_width,
        ps_rounds_comms_time,
        bottom=round_compute_time,
        width=bar_width,
        label="PS" if i == 1 else "",
        # color="lightgray",
        color=color_palette[-4],
        edgecolor="black",
        zorder=3,
        hatch=patterns[8],
    )

    # Add percentage text on top of the bars
    plt.text(
        (i + 1) - bar_width,
        round_compute_time + rar_rounds_comms_time + 100,
        f"{(rar_rounds_comms_time / (round_compute_time + rar_rounds_comms_time)) * 100:.1f}%",
        ha="center",
        va="bottom",
        fontsize=12,
        zorder=4,
    )
    plt.text(
        (i + 1),
        round_compute_time + ar_rounds_comms_time + 100,
        f"{(ar_rounds_comms_time / (round_compute_time + ar_rounds_comms_time)) * 100:.1f}%",
        ha="center",
        va="bottom",
        fontsize=12,
        zorder=4,
    )
    plt.text(
        (i + 1) + bar_width,
        round_compute_time + ps_rounds_comms_time + 100,
        f"{(ps_rounds_comms_time / (round_compute_time + ps_rounds_comms_time)) * 100:.1f}%",
        ha="center",
        va="bottom",
        fontsize=12,
        zorder=4,
    )

# Customize the legend to assign the color to the number of steps and the line styles to the perplexity value
plt.legend(
    loc="upper left",
    bbox_to_anchor=(0.05, 0.5),  # Adjust this tuple to move the legend
    fontsize="small",
    title_fontsize="medium",
    fancybox=True,
    framealpha=0.8,
    ncol=1,
)
plt.yscale("log")
plt.xticks(
    [2, 3, 4, 5],
    labels=["2", "4", "8", "16"],
)
plt.xlabel("Number of Clients")
plt.ylabel("Wall Time (s)")
plt.ylim(8_000, 10_200)
plt.yticks(
    [8_000, 9_000, 10_000, 12_000],
    labels=[
        r"$8_{\times10^3}$",
        r"$9_{\times10^3}$",
        r"$1_{\times10^4}$",
        r"$1.2_{\times10^4}$",
    ],
)
plt.savefig("wall_time_split_bar_plot_128steps.pdf", bbox_inches="tight", dpi=dpi)
plt.show()

In [ ]:
# Load the WandB data
experiment_type = "full_participation"
experiment_hp = "64_steps"
min_ppl = 35  # Cherry-picked value
bar_width = 0.3  # Width of each bar
fig = plt.figure(figsize=fig_size1, dpi=dpi)

# Set grid lines with lower zorder to ensure they are in the background
plt.grid(axis="y", which="minor", lw=0.5, zorder=0)
plt.grid(axis="y", which="major", zorder=0)
y_coordinates: list[int] = []
x_coordinates: list[float] = []
conf_values = sorted([int(x) for x in run_id_dict[experiment_type][experiment_hp]])
j = 0
for i, conf in enumerate(conf_values):
    if conf == 1:
        continue
    run_uuid = run_id_dict[experiment_type][experiment_hp][str(conf)][0]
    try:
        server_metrics_df = pd.read_pickle(f"server_{run_uuid}.pkl")
        client_metrics_df = pd.read_pickle(f"clients_{run_uuid}.pkl")
    except FileNotFoundError:
        server_metrics_df, client_metrics_df = download_metrics(
            *run_id_dict[experiment_type][experiment_hp][str(conf)]
        )
        # Dump the dataframes to pickle files
        server_metrics_df.to_pickle(f"server_{run_uuid}.pkl")
        client_metrics_df.to_pickle(f"clients_{run_uuid}.pkl")
    steps_to_min, new_min_ppl = find_federated_round_for_perplexity(
        client_metrics_df=client_metrics_df,
        perplexity_value=min_ppl,
        rolling_window=100,
    )
    # Initialize the model
    n_local_steps = int(experiment_hp.replace("_steps", ""))
    n_rounds = steps_to_min // n_local_steps
    model = WallTimeModel(
        num_rounds=n_rounds,
        num_clients_per_round=int(conf),
        num_parameters=125_000_000,
        local_steps=n_local_steps,
        local_throughput=2,  # 2 batches per second
    )

    # Calculate total wall time for different approaches
    round_compute_time = model.num_rounds * model.client_training_time()
    rar_rounds_comms_time = model.num_rounds * model.ring_allreduce_communication_time()
    ar_rounds_comms_time = model.num_rounds * model.allreduce_communication_time()
    ps_rounds_comms_time = (
        model.num_rounds * model.parameter_server_communication_time()
    )

    # Construct adjacent bars for each configuration
    plt.bar(
        (i + 1) - bar_width,
        round_compute_time,
        width=bar_width,
        # color="gray",
        color=color_palette[-1],
        edgecolor="black",
        label="LC" if j == 0 else "",
        zorder=3,
    )
    j += 1
    plt.bar(
        (i + 1) - bar_width,
        rar_rounds_comms_time,
        bottom=round_compute_time,
        width=bar_width,
        label="RAR" if i == 1 else "",
        # color="lightgray",
        color=color_palette[-2],
        edgecolor="black",
        zorder=3,
        hatch=patterns[6],
    )
    plt.bar(
        (i + 1),
        round_compute_time,
        width=bar_width,
        # color="gray",
        color=color_palette[-1],
        edgecolor="black",
        zorder=3,
    )
    plt.bar(
        (i + 1),
        ar_rounds_comms_time,
        bottom=round_compute_time,
        width=bar_width,
        label="AR" if i == 1 else "",
        # color="lightgray",
        color=color_palette[-3],
        edgecolor="black",
        zorder=3,
        hatch=patterns[7],
    )
    plt.bar(
        (i + 1) + bar_width,
        round_compute_time,
        width=bar_width,
        # color="gray",
        color=color_palette[-1],
        edgecolor="black",
        zorder=3,
    )
    plt.bar(
        (i + 1) + bar_width,
        ps_rounds_comms_time,
        bottom=round_compute_time,
        width=bar_width,
        label="PS" if i == 1 else "",
        # color="lightgray",
        color=color_palette[-4],
        edgecolor="black",
        zorder=3,
        hatch=patterns[8],
    )

    # Add percentage text on top of the bars
    plt.text(
        (i + 1) - bar_width,
        round_compute_time + rar_rounds_comms_time + 100,
        f"{(rar_rounds_comms_time / (round_compute_time + rar_rounds_comms_time)) * 100:.1f}%",
        ha="center",
        va="bottom",
        fontsize=12,
        zorder=4,
    )
    plt.text(
        (i + 1),
        round_compute_time + ar_rounds_comms_time + 100,
        f"{(ar_rounds_comms_time / (round_compute_time + ar_rounds_comms_time)) * 100:.1f}%",
        ha="center",
        va="bottom",
        fontsize=12,
        zorder=4,
    )
    plt.text(
        (i + 1) + bar_width,
        round_compute_time + ps_rounds_comms_time + 100,
        f"{(ps_rounds_comms_time / (round_compute_time + ps_rounds_comms_time)) * 100:.1f}%",
        ha="center",
        va="bottom",
        fontsize=12,
        zorder=4,
    )

# Customize the legend to assign the color to the number of steps and the line styles to the perplexity value
plt.legend(
    loc="upper left",
    bbox_to_anchor=(0.05, 0.47),  # Adjust this tuple to move the legend
    fontsize="small",
    title_fontsize="medium",
    fancybox=True,
    framealpha=0.8,
    ncol=1,
)
plt.yscale("log")
plt.xticks(
    [2, 3, 4, 5],
    labels=["2", "4", "8", "16"],
)
plt.xlabel("Number of Clients")
plt.ylabel("Wall Time (s)")
plt.ylim(7_000, 10_200)
plt.yticks(
    [7_000, 8_000, 9_000, 10_000, 12_000, 14_000, 14_800],
    labels=[
        r"$7_{\times10^3}$",
        r"$8_{\times10^3}$",
        r"$9_{\times10^3}$",
        r"$1_{\times10^4}$",
        r"$1.2_{\times10^4}$",
        r"$1.4_{\times10^4}$",
        "",
    ],
)
plt.savefig("wall_time_split_bar_plot_64steps.pdf", bbox_inches="tight", dpi=dpi)
plt.show()

In [ ]:
SERVER_BANDWIDTH = 125  # 125 MBps == 1Gbps
SERVER_BANDWIDTH = 1_250  # 10 Gbps
SERVER_BANDWIDTH1 = 12_500  # 100 Gbps -- one link slow RoCE/IB
SERVER_BANDWIDTH = 50_000  # 400 Gbps -- one link fast RoCE/IB
SERVER_BANDWIDTH = 200_000  # 1600 Gbps -- four link fast RoCE/IB
SERVER_BANDWIDTH = 250  # 250 MBps == 2Gbps
SERVER_BANDWIDTH = 312.5  # 312.5 MBps == 2.5Gbps
SERVER_BANDWIDTH = 625  # 625 MBps == 5Gbps
SERVER_BANDWIDTH = 1_250  # 10 Gbps
SERVER_BANDWIDTH1 = SERVER_BANDWIDTH

In [ ]:
# Going forward in the convergence curve

# fed 1B min PPL: 18.26742 @ 9700 total steps
# cen 1B PPL: 18.26171 @ 19759 total steps

# fed 3B min PPL: 16.13603 @ 13000 total steps
# cen 3B PPL: 16.13357 @ 22900 total steps

# fed 7B min PPL: 14.04218 @ 11000 total steps
# cen 7B PPL: 14.03865 @ 21908 total steps

# Going backward in the convergence curve

# fed 1B min PPL: 21.30412 @ 6000 total steps
# cen 1B PPL: 21.31308 @ 10000 total steps

# fed 3B min PPL: 18.56525 @ 7500 total steps
# cen 3B PPL: 18.319 @ 13000 total steps

# fed 7B min PPL: 16.62725 @ 6000 total steps
# cen 7B PPL: 16.35467 @ 11000 total steps

# Fixing to PPL at around 21

# fed 7B min PPL: 20.95154 @ 4000 total steps
# cen 7B PPL: 21.24964 @ 4900 total steps

# fed 3B min PPL: 21.05414 @ 5500 total steps
# cen 3B PPL: 21.30629 @ 7162 total steps

In [ ]:
# DDP 4xH100, cen-bench-1B-bs64-20241030_091814_centralised,
# FED: local MFU (pre device) 1.1245, avg 83% GPU utilization locally
# CEN: local MFU (pre device) 0.8027, avg 74% GPU utilization locally
global_batch_size = 512
local_batch_size = 8
gradient_accumulation_steps = 8
single_worker_client_throughput = 0.8396  # batches per second
n_workers_clients = global_batch_size // (
    local_batch_size * gradient_accumulation_steps
)
# single_worker_client_throughput = 0.8396  # batches per second
fed_single_worker_client_throughput = 0.147  # batches per second
cen_single_worker_client_throughput = 0.8396  # batches per second
n_rounds = 19
n_local_steps = 500
fed_1b = WallTimeModel(
    num_rounds=n_rounds,
    num_clients_per_round=n_workers_clients,
    num_parameters=1_315_950_592,
    local_steps=n_local_steps,
    local_throughput=fed_single_worker_client_throughput,
    server_bandwidth=SERVER_BANDWIDTH,
)
round_compute_time = fed_1b.num_rounds * fed_1b.client_training_time() / 3600
rar_rounds_comms_time = (
    fed_1b.num_rounds * fed_1b.ring_allreduce_communication_time() / 3600
)
ar_rounds_comms_time = fed_1b.num_rounds * fed_1b.allreduce_communication_time() / 3600
ps_rounds_comms_time = (
    fed_1b.num_rounds * fed_1b.parameter_server_communication_time() / 3600
)
log(INFO, f"Fed-1B::Round compute time: {round_compute_time}")
log(INFO, f"Fed-1B::RAR rounds comms time: {rar_rounds_comms_time}")
log(INFO, f"Fed-1B::AR rounds comms time: {ar_rounds_comms_time}")
log(INFO, f"Fed-1B::PS rounds comms time: {ps_rounds_comms_time}")
cen_1b = WallTimeModel(
    num_rounds=n_rounds * n_local_steps + 10_259,
    num_clients_per_round=n_workers_clients,
    num_parameters=1_315_950_592,
    local_steps=1,
    local_throughput=cen_single_worker_client_throughput,
    server_bandwidth=SERVER_BANDWIDTH1,
)
round_compute_time = cen_1b.num_rounds * cen_1b.client_training_time() / 3600
rar_rounds_comms_time = (
    cen_1b.num_rounds * cen_1b.ring_allreduce_communication_time() / 3600
)
ar_rounds_comms_time = cen_1b.num_rounds * cen_1b.allreduce_communication_time() / 3600
ps_rounds_comms_time = (
    cen_1b.num_rounds * cen_1b.parameter_server_communication_time() / 3600
)
log(INFO, f"Cen-1B::Round compute time: {round_compute_time}")
log(INFO, f"Cen-1B::RAR rounds comms time: {rar_rounds_comms_time}")
log(INFO, f"Cen-1B::AR rounds comms time: {ar_rounds_comms_time}")
log(INFO, f"Cen-1B::PS rounds comms time: {ps_rounds_comms_time}")

In [ ]:
# FSDP-FS-AC 8xH100, cen-bench-3B-fsdp-fs-bs128-20241029_224025_centralised,
# FED: local MFU (per device) 0.240, avg 78% GPU utilization locally
# CEN: local MFU (per device) 0.165, avg 81% GPU utilization locally
global_batch_size = 512
local_batch_size = 16
gradient_accumulation_steps = 8
n_workers_clients = global_batch_size // (
    local_batch_size * gradient_accumulation_steps
)
# single_worker_client_throughput = 0.395  # batches per second
fed_single_worker_client_throughput = 0.144  # batches per second
cen_single_worker_client_throughput = 0.395  # batches per second
n_rounds = 26
n_local_steps = 500
fed_3b = WallTimeModel(
    num_rounds=n_rounds,
    num_clients_per_round=n_workers_clients,
    num_parameters=2_651_837_440,
    local_steps=n_local_steps,
    local_throughput=fed_single_worker_client_throughput,
    server_bandwidth=SERVER_BANDWIDTH,
)
round_compute_time = fed_3b.num_rounds * fed_3b.client_training_time() / 3600
rar_rounds_comms_time = (
    fed_3b.num_rounds * fed_3b.ring_allreduce_communication_time() / 3600
)
ar_rounds_comms_time = fed_3b.num_rounds * fed_3b.allreduce_communication_time() / 3600
ps_rounds_comms_time = (
    fed_3b.num_rounds * fed_3b.parameter_server_communication_time() / 3600
)
log(INFO, f"Fed-3B::Round compute time: {round_compute_time}")
log(INFO, f"Fed-3B::RAR rounds comms time: {rar_rounds_comms_time}")
log(INFO, f"Fed-3B::AR rounds comms time: {ar_rounds_comms_time}")
log(INFO, f"Fed-3B::PS rounds comms time: {ps_rounds_comms_time}")
cen_3b = WallTimeModel(
    num_rounds=n_rounds * n_local_steps + 9_900,
    num_clients_per_round=n_workers_clients,
    num_parameters=2_651_837_440,
    local_steps=1,
    local_throughput=cen_single_worker_client_throughput,
    server_bandwidth=SERVER_BANDWIDTH1,
)
round_compute_time = cen_3b.num_rounds * cen_3b.client_training_time() / 3600
rar_rounds_comms_time = (
    cen_3b.num_rounds * cen_3b.ring_allreduce_communication_time() / 3600
)
ar_rounds_comms_time = cen_3b.num_rounds * cen_3b.allreduce_communication_time() / 3600
ps_rounds_comms_time = (
    cen_3b.num_rounds * cen_3b.parameter_server_communication_time() / 3600
)
log(INFO, f"Cen-3B::Round compute time: {round_compute_time}")
log(INFO, f"Cen-3B::RAR rounds comms time: {rar_rounds_comms_time}")
log(INFO, f"Cen-3B::AR rounds comms time: {ar_rounds_comms_time}")
log(INFO, f"Cen-3B::PS rounds comms time: {ps_rounds_comms_time}")

In [ ]:
# FSDP-FS-AC 8xH100, cen-bench-7B-fsdp-fs-4-2-bs256-20241029_222936_centralised,
# FED: local MFU (per device) 0.224, avg 90% GPU utilization locally
# CEN: local MFU (per device) 0.335, avg 88% GPU utilization locally
global_batch_size = 1024
local_batch_size = 16
gradient_accumulation_steps = 16
n_workers_clients = global_batch_size // (
    local_batch_size * gradient_accumulation_steps
)
fed_single_worker_client_throughput = 0.032  # batches per second
cen_single_worker_client_throughput = 0.12  # batches per second
n_rounds = 22
n_local_steps = 500
fed_7b = WallTimeModel(
    num_rounds=n_rounds,
    num_clients_per_round=n_workers_clients,
    num_parameters=6_658_859_008,
    local_steps=n_local_steps,
    local_throughput=fed_single_worker_client_throughput,
    server_bandwidth=SERVER_BANDWIDTH,
)
round_compute_time = (
    fed_7b.num_rounds * fed_7b.client_training_time() / 3600
)  # Convert to hours
rar_rounds_comms_time = (
    fed_7b.num_rounds * fed_7b.ring_allreduce_communication_time() / 3600
)  # Convert to hours
ar_rounds_comms_time = (
    fed_7b.num_rounds * fed_7b.allreduce_communication_time() / 3600
)  # Convert to hours
ps_rounds_comms_time = (
    fed_7b.num_rounds * fed_7b.parameter_server_communication_time() / 3600
)  # Convert to hours
log(INFO, f"Fed-7B::Round compute time: {round_compute_time} hours")
log(INFO, f"Fed-7B::RAR rounds comms time: {rar_rounds_comms_time} hours")
log(INFO, f"Fed-7B::AR rounds comms time: {ar_rounds_comms_time} hours")
log(INFO, f"Fed-7B::PS rounds comms time: {ps_rounds_comms_time} hours")
cen_7b = WallTimeModel(
    num_rounds=n_rounds * n_local_steps + 10_908,
    num_clients_per_round=n_workers_clients,
    num_parameters=6_658_859_008,
    local_steps=1,
    local_throughput=cen_single_worker_client_throughput,
    server_bandwidth=SERVER_BANDWIDTH1,
)
round_compute_time = (
    cen_7b.num_rounds * cen_7b.client_training_time() / 3600
)  # Convert to hours
rar_rounds_comms_time = (
    cen_7b.num_rounds * cen_7b.ring_allreduce_communication_time() / 3600
)  # Convert to hours
ar_rounds_comms_time = (
    cen_7b.num_rounds * cen_7b.allreduce_communication_time() / 3600
)  # Convert to hours
ps_rounds_comms_time = (
    cen_7b.num_rounds * cen_7b.parameter_server_communication_time() / 3600
)  # Convert to hours
log(INFO, f"Cen-7B::Round compute time: {round_compute_time} hours")
log(INFO, f"Cen-7B::RAR rounds comms time: {rar_rounds_comms_time} hours")
log(INFO, f"Cen-7B::AR rounds comms time: {ar_rounds_comms_time} hours")
log(INFO, f"Cen-7B::PS rounds comms time: {ps_rounds_comms_time} hours")